<a href="https://colab.research.google.com/github/saher-vc/Balanced-Parentheses-Checker-/blob/main/medical_task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ================================================================
# MEDICAL QLoRA FINE-TUNING — COMPLETE WORKING CODE (April 2025)
# STEP 0: Runtime → Change runtime type → T4 GPU → Connect first!
# ================================================================

# ---------- INSTALL ----------
import subprocess, sys

print("📦 Installing packages... (3-5 mins, please wait)")
subprocess.run([sys.executable, "-m", "pip", "install",
    "unsloth", "datasets", "trl", "transformers",
    "accelerate", "bitsandbytes", "-q", "--upgrade"],
    capture_output=True)
print("✅ Packages installed!")

# ---------- IMPORTS ----------
import torch, gc, os
from unsloth import FastModel, FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
print("✅ Imports done!")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ Free memory: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ---------- CLEAR MEMORY ----------
torch.cuda.empty_cache()
gc.collect()

# ---------- LOAD MODEL (4-bit quantized) ----------
print("\n🔄 Loading model... (5-8 mins)")

model, tokenizer = FastModel.from_pretrained(
    model_name   = "unsloth/Llama-3.2-1B-Instruct",  # 1B model — fits easily on free T4!
    max_seq_length = 512,
    load_in_4bit   = True,
    load_in_8bit   = False,
    full_finetuning = False,
)
print("✅ Model loaded!")

# ---------- ADD LoRA ADAPTERS ----------
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
    max_seq_length = 512,
)
print("✅ LoRA adapters added!")
model.print_trainable_parameters()

# ---------- LOAD MEDICAL DATASET ----------
print("\n🔄 Loading medical dataset...")

dataset = load_dataset(
    "lavita/ChatDoctor-HealthCareMagic-100k",
    split = "train[:800]"   # 800 examples = ~15 mins training
)

PROMPT = """Below is a medical question. Answer it clearly and accurately.

### Question:
{}

### Answer:
{}"""

EOS = tokenizer.eos_token

def format_prompts(examples):
    texts = []
    for q, a in zip(examples["input"], examples["output"]):
        texts.append(PROMPT.format(q, a) + EOS)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print(f"✅ Dataset ready! {len(dataset)} medical Q&A examples loaded")
print(f"   Sample question: {dataset[0]['text'][:120]}...")

# ---------- CONFIGURE TRAINER ----------
trainer = SFTTrainer(
    model        = model,
    tokenizer    = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field       = "text",
        max_seq_length           = 512,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps             = 5,
        max_steps                = 60,   # 60 steps ~ 15 mins, good for internship demo
        learning_rate            = 2e-4,
        fp16                     = not torch.cuda.is_bf16_supported(),
        bf16                     = torch.cuda.is_bf16_supported(),
        logging_steps            = 5,
        output_dir               = "outputs",
        optim                    = "adamw_8bit",
        seed                     = 42,
    ),
)
print("✅ Trainer configured!")

# ---------- TRAIN ----------
print("\n🚀 Starting training... DO NOT stop or close this tab!")
print("   You will see loss numbers. They should go DOWN over time.")
print("   This takes ~15 minutes. Go grab a tea! ☕")
print("-" * 55)

trainer_stats = trainer.train()

print("-" * 55)
print("🎉 TRAINING COMPLETE!")
print(f"   Final loss : {trainer_stats.metrics['train_loss']:.4f}")
print(f"   Time taken : {trainer_stats.metrics['train_runtime']/60:.1f} minutes")

# ---------- SAVE ADAPTER ----------
print("\n💾 Saving adapter...")
model.save_pretrained("medical_lora_adapter")
tokenizer.save_pretrained("medical_lora_adapter")
print("✅ Saved locally!")

# ---------- SAVE TO GOOGLE DRIVE ----------
print("\n☁️  Saving to Google Drive (won't lose it after restart)...")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import shutil
save_path = "/content/drive/MyDrive/medical_lora_adapter"
if os.path.exists(save_path):
    shutil.rmtree(save_path)
shutil.copytree("medical_lora_adapter", save_path)
print(f"✅ Saved to Google Drive at: {save_path}")

# ---------- TEST THE MODEL ----------
print("\n🧪 Testing your fine-tuned medical model...")
FastLanguageModel.for_inference(model)

test_questions = [
    "What are the main symptoms of Type 2 Diabetes?",
    "What medications are commonly used for high blood pressure?",
]

for q in test_questions:
    inputs = tokenizer(
        [PROMPT.format(q, "")],
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        temperature    = 0.7,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("### Answer:")[-1].strip()
    print(f"\n❓ Question: {q}")
    print(f"💊 Answer  : {answer[:300]}")
    print("-" * 55)

print("\n🏆 ALL DONE! Your medical AI is fine-tuned and saved!")
print("   For your internship report:")
print("   • Model     : Llama 3.2 1B Instruct (Meta)")
print("   • Technique : QLoRA (4-bit + LoRA r=16)")
print("   • Framework : Unsloth + HuggingFace TRL")
print("   • Dataset   : ChatDoctor HealthCareMagic 100k")
print("   • Steps     : 60, lr=2e-4, batch=1, grad_accum=8")


📦 Installing packages... (3-5 mins, please wait)
✅ Packages installed!
✅ Imports done!
✅ GPU: Tesla T4
✅ Free memory: 13.1 GB

🔄 Loading model... (5-8 mins)
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

✅ Model loaded!
✅ LoRA adapters added!
trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039

🔄 Loading medical dataset...
✅ Dataset ready! 800 medical Q&A examples loaded
   Sample question: Below is a medical question. Answer it clearly and accurately.

### Question:
I woke up this morning feeling the whole r...
✅ Trainer configured!

🚀 Starting training... DO NOT stop or close this tab!
   You will see loss numbers. They should go DOWN over time.
   This takes ~15 minutes. Go grab a tea! ☕
-------------------------------------------------------


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
5,3.464868
10,3.192167
15,3.003732
20,3.007856
25,2.893880
30,2.872087
35,2.786931
40,2.845789
45,2.888565
50,2.866915


-------------------------------------------------------
🎉 TRAINING COMPLETE!
   Final loss : 2.9661
   Time taken : 2.2 minutes

💾 Saving adapter...
✅ Saved locally!

☁️  Saving to Google Drive (won't lose it after restart)...


MessageError: Error: credential propagation was unsuccessful